# Final Model Selection and Test-Set Evaluation

## Goal

Select the final forecasting model using validation evidence and evaluate it once on the previously untouched test set.

The project has compared multiple model families and two phases of feature engineering. The strongest validation model is Phase 2 Histogram Gradient Boosting.

The purpose of this notebook is to:

1. Confirm the final model-selection decision.
2. Retrain or restore the selected model using the finalized Phase 2 feature set.
3. Evaluate the selected model on the untouched test period.
4. Report final generalization metrics.
5. Compare validation and test performance.
6. Summarize the final modeling conclusion for the project.

## Final Model Selection

Based on validation performance, the selected model is:

**Phase 2 Histogram Gradient Boosting**

Validation RMSE:

`0.7867`

This model outperformed:

- persistence
- linear regression
- random forest
- Phase 1 Histogram Gradient Boosting
- Phase 1 neural network
- Huber-loss neural network
- reduced-dropout neural network
- Phase 2 neural network

## Phase 2 Neural Network Reference

The strongest Phase 2 neural-network result was:

- MAE: `0.5218`
- RMSE: `0.7973`
- R²: `0.4215`

The neural network remained competitive, but did not outperform Histogram Gradient Boosting.

## Experimental Rule

The test set has remained untouched throughout:

- feature engineering
- model comparison
- neural-network experiments
- hyperparameter decisions
- final model selection

The test set will now be used once to estimate how well the selected model generalizes to unseen future data.

## Plan

1. Load the finalized Phase 2 feature dataset.
2. Recreate the same chronological 70/15/15 split.
3. Recreate the finalized Phase 2 HGB configuration.
4. Train the final model using the established training data.
5. Generate predictions for the untouched test set.
6. Calculate:
   - MAE
   - RMSE
   - R²
7. Compare test performance with validation performance.
8. Inspect actual-vs-predicted behavior on the test period.
9. Analyze whether generalization performance remained stable.
10. Record the final project conclusion and selected model.

## Primary Question

Does the Phase 2 Histogram Gradient Boosting model maintain its validation performance when evaluated on genuinely unseen future data?

## Success Criteria

The final model does not need to reproduce the validation metrics exactly.

A successful result is one where:

- test performance remains reasonably close to validation performance
- there is no evidence of severe overfitting
- the model continues to outperform the original persistence baseline
- the final model choice remains supported by the full experimental evidence

## Final Validation Benchmark

Phase 2 HGB validation performance:

- RMSE: `0.7867`

This value will be compared with the final test-set RMSE to assess generalization.

In [1]:
## 1. Load the Final Phase 2 Dataset

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

data_path = Path(
    "../data/processed/household_power_features_phase2.parquet"
)

df = pd.read_parquet(data_path)
df = df.sort_values("datetime").reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Start:", df["datetime"].min())
print("End:", df["datetime"].max())

Dataset shape: (1950244, 33)
Start: 2006-12-17 17:23:00
End: 2010-11-26 20:02:00


In [2]:
## 2. Recreate the Final Chronological Split

n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))

print("\nTrain:")
print(train_df["datetime"].min(), "→", train_df["datetime"].max())

print("\nValidation:")
print(val_df["datetime"].min(), "→", val_df["datetime"].max())

print("\nTest:")
print(test_df["datetime"].min(), "→", test_df["datetime"].max())

Train rows: 1365170
Validation rows: 292537
Test rows: 292537

Train:
2006-12-17 17:23:00 → 2009-09-16 02:55:00

Validation:
2009-09-16 02:56:00 → 2010-04-21 22:39:00

Test:
2010-04-21 22:40:00 → 2010-11-26 20:02:00


In [3]:
## define feature set

phase2_feature_cols = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",

    "power_lag_1",
    "power_lag_5",
    "power_lag_15",
    "power_lag_30",
    "power_lag_60",
    "power_lag_120",
    "power_lag_1380",

    "hour",
    "day_of_week",
    "month",
    "is_weekend",

    "rolling_mean_15",
    "rolling_mean_60",
    "rolling_std_60",
    "rolling_mean_1440",

    "power_change_1",
    "power_change_5",
    "power_change_15",
    "power_change_60",

    "rolling_std_15",
    "rolling_max_15",
    "rolling_max_60",
    "rolling_min_60",
    "rolling_range_60"
]

target_col = "power_target_60"

print("Feature count:", len(phase2_feature_cols))

Feature count: 31


In [4]:
## training and test matrices

X_train = train_df[phase2_feature_cols]
y_train = train_df[target_col]

X_test = test_df[phase2_feature_cols]
y_test = test_df[target_col]

print("X train:", X_train.shape)
print("X test:", X_test.shape)

X train: (1365170, 31)
X test: (292537, 31)


In [6]:
## 3. Train the Selected Final Model

final_hgb = HistGradientBoostingRegressor(
    learning_rate=0.1,
    max_iter=200,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

final_hgb.fit(
    X_train,
    y_train
)

,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",200
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",1.0
,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255


In [7]:
## 4. Evaluate on the Untouched Test Set

test_predictions = final_hgb.predict(X_test)

In [8]:
test_mae = mean_absolute_error(
    y_test,
    test_predictions
)

test_rmse = mean_squared_error(
    y_test,
    test_predictions
) ** 0.5

test_r2 = r2_score(
    y_test,
    test_predictions
)

print(f"Final Test MAE:  {test_mae:.4f}")
print(f"Final Test RMSE: {test_rmse:.4f}")
print(f"Final Test R²:   {test_r2:.4f}")

Final Test MAE:  0.4712
Final Test RMSE: 0.6912
Final Test R²:   0.3594


In [9]:
## 5. Compare Validation and Test Performance

validation_rmse = 0.7867

rmse_change_pct = (
    (test_rmse - validation_rmse)
    / validation_rmse
) * 100

print(f"Validation RMSE: {validation_rmse:.4f}")
print(f"Test RMSE:       {test_rmse:.4f}")

print(
    f"Test RMSE change vs validation: "
    f"{rmse_change_pct:+.2f}%"
)


Validation RMSE: 0.7867
Test RMSE:       0.6912
Test RMSE change vs validation: -12.14%


In [10]:
# Close to 0%
# → test performance closely matches validation

# Small positive %
# → test period was somewhat harder

# Negative %
# → test period was somewhat easier

# Large positive %
# → possible distribution shift or weaker generalization

In [11]:
final_metrics = pd.DataFrame({
    "split": ["Validation", "Test"],
    "RMSE": [validation_rmse, test_rmse]
})

final_metrics

,split,RMSE
0,Validation,0.786700
1,Test,0.691174


### Final Test-Set Interpretation

The selected Phase 2 Histogram Gradient Boosting model achieved a test RMSE of
approximately `0.6912`, compared with a validation RMSE of `0.7867`.

This represents a test RMSE approximately `12.14%` lower than the validation RMSE.

The lower test error suggests that the final test period was easier to forecast
than the validation period. This does not necessarily mean the model improved after
selection; rather, the statistical characteristics of the test period likely made
the forecasting task somewhat easier.

Most importantly, the model maintained strong performance on completely unseen
future data and showed no evidence of severe overfitting.

The final result supports the earlier validation-based decision to select Phase 2
Histogram Gradient Boosting as the preferred model.